In [ ]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup
from fantasy.nba_stats import get_player_stats, get_games_this_week
from fantasy.analysis import project_player
from fantasy.llm import build_start_sit_prompt, ask_gemini
import pandas as pd

In [ ]:
matchup = get_current_matchup()
roster = get_my_roster()
week_start, week_end = matchup["week_start"], matchup["week_end"]

players = []
for p in roster:
    stats = get_player_stats(p["name"])
    games = get_games_this_week(p["team_abbr"], week_start, week_end)
    projected = project_player(stats, games, p["status"]) if stats else {}
    players.append({**p, "stats": stats, "games_remaining": games, "projected": projected})

# Sort by projected PTS descending
players.sort(key=lambda x: x["projected"].get("PTS", 0), reverse=True)

In [ ]:
rows = []
for p in players:
    rows.append({
        "Player": p["name"],
        "Pos": p["position"],
        "Status": p["status"] or "Active",
        "Games Left": p["games_remaining"],
        "Proj PTS": round(p["projected"].get("PTS", 0), 1),
        "Proj REB": round(p["projected"].get("REB", 0), 1),
        "Proj AST": round(p["projected"].get("AST", 0), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
prompt = build_start_sit_prompt(players)
advice = ask_gemini(prompt)
print("\n=== TODAY'S LINEUP CARD ===\n")
print(advice)